# Domain 2: Tool Design & MCP Integration
**Exam**: Claude Certified Architect – Foundations (CCAR-F)
**Weighting**: 18% of exam
**Depth**: deep

## Domain Overview

This domain covers designing clear, unambiguous tool interfaces, implementing structured
error responses for MCP tools, distributing tools appropriately across agents, integrating MCP
servers into Claude Code/agent workflows, and selecting built-in tools (Read, Write, Edit, Bash, Grep,
Glob) effectively.

**Why it matters**: tool selection reliability is one of the most common production failure points —
vague or overlapping tool descriptions cause agents to call the wrong tool, and poor error handling
prevents agents from recovering intelligently.

**Key concepts to know**: tool description quality, structured MCP error responses (isError,
errorCategory, isRetryable), least-privilege tool distribution, MCP server scoping, and built-in tool
selection criteria.

In [ ]:
# Setup: these notebooks demonstrate Claude Agent SDK / Claude API / MCP concepts using
# plain Python simulations of the relevant objects and control flow, so every cell is runnable
# without needing a live Claude Agent SDK installation or API key. Where call_claude() is used,
# it falls back to a simulated response if ANTHROPIC_API_KEY is not set.
import os

def call_claude(prompt, system=None, model="claude-sonnet-4-5", max_tokens=600):
    try:
        import anthropic
        api_key = os.environ.get("ANTHROPIC_API_KEY")
        if not api_key:
            raise RuntimeError("no ANTHROPIC_API_KEY set")
        client = anthropic.Anthropic(api_key=api_key)
        kwargs = {"model": model, "max_tokens": max_tokens, "messages": [{"role": "user", "content": prompt}]}
        if system:
            kwargs["system"] = system
        resp = client.messages.create(**kwargs)
        return resp.content[0].text
    except Exception:
        return f"[SIMULATED RESPONSE - set ANTHROPIC_API_KEY to call the real API] Prompt: {prompt[:120]}..."

print("call_claude() ready. Concepts below use plain Python to simulate Agent SDK / MCP / Claude Code behavior.")

## Key Concepts

### Design effective tool interfaces with clear descriptions and boundaries

Tool descriptions are the primary mechanism LLMs use for tool selection. Minimal descriptions
("Retrieves customer information") lead to unreliable selection among similar tools. Effective
descriptions state input formats, example queries, edge cases, and explicit boundaries versus
similar tools.

In [ ]:
weak_tool_description = "Retrieves order details."
strong_tool_description = (
    "lookup_order(order_id: str) -> OrderDetails. Use ONLY when the user provides an order ID "
    "(format: ORD-XXXXX) or asks about a specific existing order's status/items/shipping. "
    "Do NOT use for customer account questions - use get_customer for those."
)
print(len(weak_tool_description.split()), "words vs", len(strong_tool_description.split()), "words")
print(strong_tool_description)

### Implement structured error responses for MCP tools

The MCP `isError` flag communicates tool failure. Effective error responses distinguish
transient errors (timeouts - retryable), validation errors (bad input), business errors (policy
violations - not retryable, needs a human-readable explanation), and permission errors, rather than
returning a generic "Operation failed".

In [ ]:
def mcp_tool_error(category: str, message: str, retryable: bool) -> dict:
    return {"isError": True, "errorCategory": category, "isRetryable": retryable, "message": message}

print(mcp_tool_error("transient", "Upstream service timed out", retryable=True))
print(mcp_tool_error("business", "Refund exceeds $500 policy limit", retryable=False))
print(mcp_tool_error("validation", "order_id must match ORD-XXXXX", retryable=False))

### Distribute tools appropriately across agents and configure tool choice

Giving an agent too many tools (e.g., 18 instead of 4-5) degrades selection reliability.
Scope each agent's tools to its role, with limited cross-role tools for high-frequency needs.
`tool_choice` can be `"auto"` (model may return text), `"any"` (must call a tool, any tool), or a
forced named tool.

In [ ]:
def scoped_tools_for_role(role: str) -> list[str]:
    role_tools = {
        "synthesis_agent": ["verify_fact"],                 # scoped cross-role tool for high-frequency need
        "web_search_agent": ["search_web", "fetch_page"],
        "coordinator": ["Task"],
    }
    return role_tools.get(role, [])

print(scoped_tools_for_role("synthesis_agent"))  # NOT full web search access - least privilege

tool_choice_examples = {
    "auto": "model may respond with text instead of calling a tool",
    "any": "model must call some tool, but can choose which",
    "forced": '{"type": "tool", "name": "extract_metadata"}  # must call this specific tool first',
}
for k, v in tool_choice_examples.items():
    print(k, "->", v)

### Integrate MCP servers into Claude Code and agent workflows

Project-scoped MCP servers (`.mcp.json`) are shared via version control for team-wide
tooling; user-scoped servers (`~/.claude.json`) are personal/experimental. Environment variable
expansion (e.g., `${GITHUB_TOKEN}`) keeps credentials out of committed config. MCP **resources**
expose content catalogs to reduce exploratory tool calls.

In [ ]:
project_mcp_json = {
    "mcpServers": {
        "github": {
            "command": "npx",
            "args": ["-y", "@modelcontextprotocol/server-github"],
            "env": {"GITHUB_TOKEN": "${GITHUB_TOKEN}"}  # expanded from the environment, never committed
        }
    }
}
import json
print(json.dumps(project_mcp_json, indent=2))

### Select and apply built-in tools (Read, Write, Edit, Bash, Grep, Glob) effectively

Grep searches file *contents* (function names, error messages). Glob matches file *paths* by
pattern (e.g., `**/*.test.tsx`). Edit makes targeted modifications via unique text matching; when a
match isn't unique, fall back to Read + Write for reliability.

In [ ]:
def choose_builtin_tool(task: str) -> str:
    if "find all callers of" in task or "search for error message" in task:
        return "Grep - searching file contents"
    if "find files named" in task or "test files" in task:
        return "Glob - matching file paths by pattern"
    if "modify a specific line" in task:
        return "Edit (fallback to Read+Write if anchor text isn't unique)"
    return "Read - to load full context first"

print(choose_builtin_tool("find all callers of process_refund"))
print(choose_builtin_tool("find all test files"))

## Mini Hands-On Project

**Covers**: tool description quality, structured errors, tool distribution, MCP config.

**Scenario**: You're building an internal inventory-lookup MCP server used by 3 different Claude
apps. Design its tool interface, error handling, and scoping.

### Step 1: Write a clear tool description

In [ ]:
inventory_tool_description = (
    "check_inventory(sku: str) -> InventoryStatus. Use ONLY when given a specific SKU code "
    "(format: SKU-#####) to check current stock count and warehouse location. "
    "Do NOT use for pricing or product description questions."
)
print(inventory_tool_description)

### Step 2: Design structured error responses

In [ ]:
def check_inventory_errors(sku: str):
    if not sku.startswith("SKU-"):
        return mcp_tool_error("validation", "sku must match SKU-##### format", retryable=False)
    return {"isError": False, "stock": 42, "warehouse": "EAST-3"}

print(check_inventory_errors("bad-sku"))
print(check_inventory_errors("SKU-12345"))

### Step 3: Configure it as a shared project MCP server

In [ ]:
inventory_mcp_config = {
    "mcpServers": {
        "inventory": {"command": "node", "args": ["inventory-server.js"], "env": {"API_KEY": "${INVENTORY_API_KEY}"}}
    }
}
print(inventory_mcp_config)

## Lab Exercises

### Exercise 1: Fix an ambiguous tool pair

**Objective**: Rewrite two overlapping tool descriptions to eliminate selection ambiguity.

**Produce**: Two clearly differentiated tool descriptions.

In [ ]:
ambiguous_tools = {
    "analyze_content": "Analyzes content and returns insights.",
    "analyze_document": "Analyzes a document and returns insights.",
}
# TODO: rewrite both descriptions (and consider renaming one) so an LLM can reliably choose between them
fixed_tools = {}
print(fixed_tools)

# Expected output: Two descriptions with non-overlapping, explicit boundaries (e.g., one for web content, one for uploaded files).

In [ ]:
# STRETCH GOAL: write 3 test queries and manually reason through which tool each should route to
# given your fixed descriptions.
test_queries = ["Analyze this webpage", "Analyze this uploaded PDF", "Analyze this pasted paragraph"]
# TODO: for each, state which fixed tool it should route to, and why

# Expected output: A clear routing decision per query, justified by the new descriptions.

### Exercise 2: Structured error taxonomy

**Objective**: Extend mcp_tool_error usage to cover a permission-denied case.

**Produce**: A new error category handled correctly by a hypothetical caller.

In [ ]:
def check_permission_and_call(user_role: str, tool_name: str):
    if tool_name == "delete_record" and user_role != "admin":
        # TODO: return a structured "permission" error using mcp_tool_error, not retryable
        pass
    return {"isError": False, "result": "done"}

print(check_permission_and_call("viewer", "delete_record"))
print(check_permission_and_call("admin", "delete_record"))

# Expected output: A structured permission error for non-admins; success for admins.

In [ ]:
# STRETCH GOAL: write a caller that reads isRetryable and only retries transient errors automatically.
def call_with_retry(result: dict, retries_left=2):
    # TODO: if result["isError"] and result.get("isRetryable") and retries_left > 0, simulate a retry
    pass

# Expected output: Retry logic that only triggers for isRetryable=True errors, never for permission/business errors.

### Exercise 3: Least-privilege tool audit

**Objective**: Audit an over-provisioned agent's tool list and cut it to least-privilege.

**Produce**: A reduced tool list with justification for each removal.

In [ ]:
over_provisioned_tools = ["search_web", "fetch_page", "check_inventory", "process_refund",
                          "delete_account", "verify_fact", "send_email", "read_logs"]
agent_role = "synthesis_agent"  # only needs to combine findings and do light fact verification

# TODO: reduce to a least-privilege list appropriate for agent_role, with a removal reason per cut tool
scoped_list = []  # TODO
removed_with_reasons = []  # TODO
print(scoped_list)
print(removed_with_reasons)

# Expected output: A short scoped list (e.g., just verify_fact) with clear reasons for removing high-risk/unrelated tools.

In [ ]:
# STRETCH GOAL: propose ONE additional scoped cross-role tool that would meaningfully help this
# agent for a high-frequency need, without over-provisioning it.
proposed_addition = "..."  # TODO
justification = "..."  # TODO

# Expected output: A single well-justified addition that doesn't reintroduce the original over-provisioning problem.

### Exercise 4: MCP scoping decision

**Objective**: Decide project- vs. user-scoped for 4 new MCP servers.

**Produce**: 4 server -> scope decisions with justification.

In [ ]:
new_servers = [
    "Company-wide Jira integration",
    "My personal experimental weather API server",
    "Shared GitHub integration for the whole team",
    "A one-off local debugging tool only I use",
]
# TODO: classify each as "project (.mcp.json)" or "user (~/.claude.json)" with a justification
for s in new_servers:
    scope = "..."  # TODO
    reason = "..."  # TODO
    print(s, "->", scope, "--", reason)

# Expected output: Team-shared servers scoped to project; personal/experimental ones scoped to user.

In [ ]:
# STRETCH GOAL: write the actual .mcp.json snippet for the "Shared GitHub integration" entry,
# including environment variable expansion for a token.
github_entry = {}  # TODO
print(github_entry)

# Expected output: A valid-looking .mcp.json entry with a ${TOKEN}-style environment variable reference.

### Exercise 5: Built-in tool selection drill

**Objective**: Choose the correct built-in tool for 5 new tasks.

**Produce**: 5 task -> tool mappings with one-sentence reasoning.

In [ ]:
tasks = [
    "Find every place calc_discount() is called across the repo",
    "Find all files ending in .config.json",
    "Change one specific line in a file where the anchor text appears twice",
    "Understand a whole module before making any changes",
    "Rename a variable that appears in exactly one unique context",
]
# TODO: choose Grep / Glob / Read+Write / Read / Edit for each, with reasoning
for t in tasks:
    tool = "..."  # TODO
    reason = "..."  # TODO
    print(t, "->", tool, "--", reason)

# Expected output: Correct tool per task, notably choosing Read+Write (not Edit) for the non-unique anchor text case.

In [ ]:
# STRETCH GOAL: write the actual choose_builtin_tool-style function extended to cover these 5 new task phrasings.
def choose_builtin_tool_v2(task: str) -> str:
    pass
for t in tasks:
    print(choose_builtin_tool_v2(t))

# Expected output: A generalized function producing the same correct tool choices programmatically.

## Self-Assessment: Tool Design & MCP Integration

Answer each question, then check the Answers cell below.


**Q1.** Why do minimal tool descriptions like 'Retrieves customer information' cause selection reliability problems?

**Q2.** What are the four error categories a well-designed MCP tool error response should distinguish?

**Q3.** Why does giving a subagent 18 tools instead of 4-5 degrade its reliability?

**Q4.** When should an MCP server be configured at the project scope vs. the user scope?

**Q5.** When should Edit be avoided in favor of Read + Write?

## Answers


**A1.** Because tool descriptions are the LLM's primary signal for tool selection; without detail, it can't reliably differentiate similar tools.

**A2.** Transient, validation, business (policy), and permission errors.

**A3.** It increases decision complexity, making the agent more likely to select the wrong tool or misuse tools outside its specialization.

**A4.** Project scope (.mcp.json) for shared team tooling checked into version control; user scope (~/.claude.json) for personal/experimental servers.

**A5.** When the anchor text Edit needs to match uniquely is not actually unique in the file, risking an incorrect or failed edit.

---
**Next**: [Domain 3: Claude Code Configuration & Workflows](./03-claude-code-configuration-and-workflows.ipynb)
**Back**: [README](./README.md)